In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# Project 13 Dataset Builder
# Amazon Seller Performance Audit
# =========================

np.random.seed(42)

# -------------------------
# 1. Setup
# -------------------------
output_folder = Path("project_13_amazon_seller_audit")
output_folder.mkdir(exist_ok=True)

start_date = "2025-01-01"
end_date = "2025-06-30"
dates = pd.date_range(start=start_date, end=end_date, freq="D")

n_products = 100

categories = {
    "Home & Kitchen": [
        "Storage Basket", "Knife Set", "Water Bottle", "Air Fryer Liner",
        "Coffee Grinder", "Food Container", "Dish Rack", "LED Lamp"
    ],
    "Fitness": [
        "Resistance Band", "Yoga Mat", "Protein Shaker", "Foam Roller",
        "Skipping Rope", "Ankle Weights", "Gym Gloves", "Massage Ball"
    ],
    "Electronics": [
        "USB Hub", "Bluetooth Speaker", "Phone Stand", "Wireless Charger",
        "Laptop Sleeve", "Desk Fan", "Mini Projector", "Webcam Cover"
    ],
    "Beauty": [
        "Face Serum", "Hair Brush", "Makeup Organiser", "Skin Cream",
        "LED Mirror", "Scalp Massager", "Cotton Pads", "Travel Bottle Set"
    ],
    "Pet Supplies": [
        "Pet Bowl", "Dog Toy", "Cat Brush", "Lint Roller",
        "Pet Blanket", "Training Pads", "Harness", "Treat Pouch"
    ]
}

category_list = list(categories.keys())

# -------------------------
# 2. Products table
# -------------------------
product_rows = []

for i in range(1, n_products + 1):
    category = np.random.choice(category_list)

    # Create product name
    base_name = np.random.choice(categories[category])
    product_name = f"{base_name} {np.random.choice(['Basic', 'Plus', 'Pro', 'Max', 'Lite'])}"

    # Assign product tier to vary economics and demand
    tier = np.random.choice(
        ["Budget", "Mid", "Premium"],
        p=[0.45, 0.4, 0.15]
    )

    if tier == "Budget":
        selling_price = round(np.random.uniform(8, 22), 2)
    elif tier == "Mid":
        selling_price = round(np.random.uniform(22, 55), 2)
    else:
        selling_price = round(np.random.uniform(55, 150), 2)

    # Random margin structure, not uniform
    target_margin_pct = np.random.uniform(0.25, 0.60)
    unit_cost = round(selling_price * (1 - target_margin_pct), 2)

    # Avoid absurdly tiny cost
    unit_cost = max(2.00, unit_cost)

    # Base demand level by tier/category mix
    if tier == "Budget":
        base_daily_units = np.random.uniform(4, 16)
    elif tier == "Mid":
        base_daily_units = np.random.uniform(2, 10)
    else:
        base_daily_units = np.random.uniform(0.5, 4.5)

    # Product-specific ad behaviour
    ad_strategy = np.random.choice(
        ["Efficient", "Average", "Wasteful", "Underinvested"],
        p=[0.25, 0.4, 0.2, 0.15]
    )

    # Conversion and click tendency
    base_ctr = np.random.uniform(0.012, 0.055)  # click-through rate
    base_conversion = np.random.uniform(0.03, 0.18)  # clicks to units proxy
    base_cpc = np.random.uniform(0.15, 1.20)

    # Some category uplift effects
    category_seasonality_strength = np.random.uniform(0.05, 0.30)

    product_rows.append({
        "product_id": f"SKU{i:03d}",
        "product_name": product_name,
        "category": category,
        "tier": tier,
        "selling_price": selling_price,
        "unit_cost": unit_cost,
        "base_margin_pct": round((selling_price - unit_cost) / selling_price, 4),
        "base_daily_units": round(base_daily_units, 2),
        "ad_strategy": ad_strategy,
        "base_ctr": round(base_ctr, 4),
        "base_conversion": round(base_conversion, 4),
        "base_cpc": round(base_cpc, 2),
        "seasonality_strength": round(category_seasonality_strength, 3)
    })

products = pd.DataFrame(product_rows)

# -------------------------
# 3. Daily sales + ads generation
# -------------------------
daily_rows = []

for _, p in products.iterrows():
    for date in dates:
        month = date.month
        day_of_week = date.dayofweek  # Monday=0, Sunday=6

        # Mild weekly pattern
        weekend_multiplier = 1.08 if day_of_week in [5, 6] else 1.0

        # Category/month seasonality
        seasonality = 1.0
        if p["category"] == "Fitness" and month in [1, 2, 3]:
            seasonality += p["seasonality_strength"]
        elif p["category"] == "Home & Kitchen" and month in [4, 5]:
            seasonality += p["seasonality_strength"] * 0.8
        elif p["category"] == "Beauty" and month in [2, 3, 6]:
            seasonality += p["seasonality_strength"] * 0.7
        elif p["category"] == "Pet Supplies" and month in [1, 4]:
            seasonality += p["seasonality_strength"] * 0.6
        elif p["category"] == "Electronics" and month in [5, 6]:
            seasonality += p["seasonality_strength"] * 0.9

        # Random daily demand noise
        demand_noise = np.random.normal(1.0, 0.22)
        demand_noise = max(0.35, demand_noise)

        expected_units = p["base_daily_units"] * weekend_multiplier * seasonality * demand_noise
        units_sold = max(0, int(np.random.poisson(expected_units)))

        # Random discounting/promotions on some days
        promo_flag = np.random.choice([0, 1], p=[0.9, 0.1])
        if promo_flag == 1:
            discount_pct = np.random.uniform(0.05, 0.18)
        else:
            discount_pct = 0.0

        realised_selling_price = round(p["selling_price"] * (1 - discount_pct), 2)

        revenue = round(units_sold * realised_selling_price, 2)
        product_cost = round(units_sold * p["unit_cost"], 2)

        # Advertising behaviour by strategy
        if p["ad_strategy"] == "Efficient":
            impressions = max(0, int(np.random.normal(1800, 500)))
            ctr = p["base_ctr"] * np.random.uniform(1.0, 1.25)
            cpc = p["base_cpc"] * np.random.uniform(0.85, 1.0)
            conversion = p["base_conversion"] * np.random.uniform(1.0, 1.25)
        elif p["ad_strategy"] == "Average":
            impressions = max(0, int(np.random.normal(2200, 700)))
            ctr = p["base_ctr"] * np.random.uniform(0.85, 1.1)
            cpc = p["base_cpc"] * np.random.uniform(0.95, 1.1)
            conversion = p["base_conversion"] * np.random.uniform(0.85, 1.05)
        elif p["ad_strategy"] == "Wasteful":
            impressions = max(0, int(np.random.normal(2800, 900)))
            ctr = p["base_ctr"] * np.random.uniform(0.75, 0.95)
            cpc = p["base_cpc"] * np.random.uniform(1.00, 1.20)
            conversion = p["base_conversion"] * np.random.uniform(0.65, 0.90)
        else:  # Underinvested
            impressions = max(0, int(np.random.normal(900, 250)))
            ctr = p["base_ctr"] * np.random.uniform(0.95, 1.15)
            cpc = p["base_cpc"] * np.random.uniform(0.9, 1.05)
            conversion = p["base_conversion"] * np.random.uniform(0.95, 1.15)

        clicks = max(0, int(np.random.binomial(impressions, min(ctr, 0.25))))
        ad_spend = round(clicks * cpc, 2)

        # Estimated ad-attributed orders for conversion diagnostics
        ad_attributed_units = int(np.random.binomial(clicks, min(conversion, 0.35))) if clicks > 0 else 0

        # Add a small fulfilment / platform fee per unit to make economics more realistic
        fulfilment_fee_per_unit = np.random.uniform(0.8, 2.8)
        fulfilment_fees = round(units_sold * fulfilment_fee_per_unit, 2)

        # Total profit after ad spend and fulfilment fees
        profit = round(revenue - product_cost - ad_spend - fulfilment_fees, 2)

        daily_rows.append({
            "date": date,
            "product_id": p["product_id"],
            "units_sold": units_sold,
            "promo_flag": promo_flag,
            "discount_pct": round(discount_pct, 4),
            "realised_selling_price": realised_selling_price,
            "revenue": revenue,
            "product_cost": product_cost,
            "impressions": impressions,
            "clicks": clicks,
            "ad_spend": ad_spend,
            "ad_attributed_units": ad_attributed_units,
            "fulfilment_fees": fulfilment_fees,
            "profit": profit
        })

daily_performance = pd.DataFrame(daily_rows)

# -------------------------
# 4. Derived metrics
# -------------------------
daily_performance["cpc"] = np.where(
    daily_performance["clicks"] > 0,
    daily_performance["ad_spend"] / daily_performance["clicks"],
    0
)

daily_performance["roas"] = np.where(
    daily_performance["ad_spend"] > 0,
    daily_performance["revenue"] / daily_performance["ad_spend"],
    np.nan
)

daily_performance["acos"] = np.where(
    daily_performance["revenue"] > 0,
    daily_performance["ad_spend"] / daily_performance["revenue"],
    np.nan
)

daily_performance["profit_margin_pct"] = np.where(
    daily_performance["revenue"] > 0,
    daily_performance["profit"] / daily_performance["revenue"],
    np.nan
)

daily_performance["click_to_order_proxy"] = np.where(
    daily_performance["clicks"] > 0,
    daily_performance["ad_attributed_units"] / daily_performance["clicks"],
    np.nan
)

# -------------------------
# 5. SKU summary table
# -------------------------
sku_summary = (
    daily_performance
    .groupby("product_id", as_index=False)
    .agg({
        "units_sold": "sum",
        "revenue": "sum",
        "product_cost": "sum",
        "impressions": "sum",
        "clicks": "sum",
        "ad_spend": "sum",
        "ad_attributed_units": "sum",
        "fulfilment_fees": "sum",
        "profit": "sum"
    })
    .merge(
        products[[
            "product_id", "product_name", "category", "tier",
            "selling_price", "unit_cost", "ad_strategy"
        ]],
        on="product_id",
        how="left"
    )
)

sku_summary["cpc"] = np.where(
    sku_summary["clicks"] > 0,
    sku_summary["ad_spend"] / sku_summary["clicks"],
    0
)

sku_summary["roas"] = np.where(
    sku_summary["ad_spend"] > 0,
    sku_summary["revenue"] / sku_summary["ad_spend"],
    np.nan
)

sku_summary["acos"] = np.where(
    sku_summary["revenue"] > 0,
    sku_summary["ad_spend"] / sku_summary["revenue"],
    np.nan
)

sku_summary["profit_margin_pct"] = np.where(
    sku_summary["revenue"] > 0,
    sku_summary["profit"] / sku_summary["revenue"],
    np.nan
)

sku_summary["click_to_order_proxy"] = np.where(
    sku_summary["clicks"] > 0,
    sku_summary["ad_attributed_units"] / sku_summary["clicks"],
    np.nan
)

# -------------------------
# 6. Add waste / scale flags
# -------------------------
# Waste = high spend but weak returns / low conversion
sku_summary["waste_flag"] = np.where(
    (sku_summary["ad_spend"] > sku_summary["ad_spend"].quantile(0.70)) &
    (
        (sku_summary["roas"] < sku_summary["roas"].quantile(0.30)) |
        (sku_summary["click_to_order_proxy"] < sku_summary["click_to_order_proxy"].quantile(0.30)) |
        (sku_summary["profit"] < 0)
    ),
    1, 0
)

# Scale = strong returns, strong profit, not already maxed on spend
sku_summary["scale_flag"] = np.where(
    (sku_summary["roas"] > sku_summary["roas"].quantile(0.75)) &
    (sku_summary["profit"] > sku_summary["profit"].quantile(0.65)) &
    (sku_summary["ad_spend"] < sku_summary["ad_spend"].quantile(0.60)),
    1, 0
)

# Margin pressure flag
sku_summary["margin_pressure_flag"] = np.where(
    (sku_summary["profit_margin_pct"] < 0.08) & (sku_summary["revenue"] > sku_summary["revenue"].median()),
    1, 0
)

# -------------------------
# 7. Separate tables for sales and ads if wanted
# -------------------------
sales = daily_performance[[
    "date", "product_id", "units_sold", "promo_flag",
    "discount_pct", "realised_selling_price", "revenue",
    "product_cost", "fulfilment_fees", "profit"
]].copy()

ads = daily_performance[[
    "date", "product_id", "impressions", "clicks",
    "ad_spend", "ad_attributed_units", "cpc", "roas",
    "acos", "click_to_order_proxy"
]].copy()

# -------------------------
# 8. Save outputs
# -------------------------
products.to_csv(output_folder / "products.csv", index=False)
sales.to_csv(output_folder / "sales.csv", index=False)
ads.to_csv(output_folder / "ads.csv", index=False)
daily_performance.to_csv(output_folder / "daily_performance.csv", index=False)
sku_summary.to_csv(output_folder / "sku_summary.csv", index=False)

print("Files created successfully:")
print(f"- {output_folder / 'products.csv'}")
print(f"- {output_folder / 'sales.csv'}")
print(f"- {output_folder / 'ads.csv'}")
print(f"- {output_folder / 'daily_performance.csv'}")
print(f"- {output_folder / 'sku_summary.csv'}")

print("\nQuick checks:")
print(f"Products: {products.shape}")
print(f"Sales rows: {sales.shape}")
print(f"Ads rows: {ads.shape}")
print(f"Daily performance rows: {daily_performance.shape}")
print(f"SKU summary rows: {sku_summary.shape}")

print("\nSample SKU summary:")
print(
    sku_summary[[
        "product_id", "product_name", "category", "revenue", "profit",
        "ad_spend", "roas", "acos", "profit_margin_pct",
        "waste_flag", "scale_flag"
    ]].head(10)
)

Files created successfully:
- project_13_amazon_seller_audit\products.csv
- project_13_amazon_seller_audit\sales.csv
- project_13_amazon_seller_audit\ads.csv
- project_13_amazon_seller_audit\daily_performance.csv
- project_13_amazon_seller_audit\sku_summary.csv

Quick checks:
Products: (100, 13)
Sales rows: (18100, 10)
Ads rows: (18100, 10)
Daily performance rows: (18100, 19)
SKU summary rows: (100, 24)

Sample SKU summary:
  product_id          product_name        category    revenue    profit  \
0     SKU001        LED Mirror Pro          Beauty   20672.74  -1718.41   
1     SKU002   Scalp Massager Plus          Beauty   21509.99    984.54   
2     SKU003  Wireless Charger Max     Electronics   15058.37 -12894.34   
3     SKU004           Dog Toy Max    Pet Supplies   48486.43   7999.43   
4     SKU005      Foam Roller Lite         Fitness  102042.16  49717.48   
5     SKU006      Treat Pouch Lite    Pet Supplies   28602.38   6284.91   
6     SKU007   Air Fryer Liner Pro  Home & Kitc

In [2]:
# 1. Overall summary
daily_performance[["revenue", "profit", "ad_spend", "roas", "acos", "profit_margin_pct"]].describe()

,revenue,profit,ad_spend,roas,acos,profit_margin_pct
count,18100.000000,18100.000000,18100.000000,18093.000000,17633.000000,17633.000000
mean,213.629348,27.406511,49.325093,9.390349,0.388347,-0.049019
std,156.727816,81.926991,38.984239,17.686898,0.569174,0.589619
min,0.000000,-230.860000,0.000000,0.000000,0.000000,-14.025578
25%,102.240000,-20.542500,20.347500,2.059279,0.097499,-0.157746
50%,179.160000,13.290000,38.315000,4.352805,0.221037,0.095775
75%,286.440000,62.062500,66.350000,9.944369,0.456231,0.259207
max,1672.320000,896.800000,302.800000,626.231884,14.492574,0.565396


In [3]:
# 2. Count waste and scale SKUs
sku_summary[["waste_flag", "scale_flag", "margin_pressure_flag"]].sum()

waste_flag              25
scale_flag              21
margin_pressure_flag    12
dtype: int64

In [7]:
# 3. Check top and bottom profit SKUs
sku_summary.sort_values("profit", ascending=False)[["product_id", "product_name", "revenue", "profit", "roas", "acos"]].head(10)

sku_summary.sort_values("profit", ascending=True)[["product_id", "product_name", "revenue", "profit", "roas", "acos"]].head(10)

,product_id,product_name,revenue,profit,roas,acos
4,SKU005,Foam Roller Lite,102042.16,49717.48,13.838101,0.072264
94,SKU095,Mini Projector Lite,106171.89,41150.02,9.563355,0.104566
50,SKU051,Pet Blanket Plus,93697.20,35531.18,22.764306,0.043928
9,SKU010,Laptop Sleeve Max,82025.17,32558.36,13.374963,0.074767
41,SKU042,Foam Roller Max,72134.07,31625.26,30.907228,0.032355
22,SKU023,Webcam Cover Plus,66785.76,26478.06,16.897478,0.059180
51,SKU052,Food Container Basic,60687.41,22873.41,29.908388,0.033435
16,SKU017,Massage Ball Basic,58455.13,22305.24,6.581243,0.151947
64,SKU065,Coffee Grinder Max,53031.09,21294.65,13.833379,0.072289
54,SKU055,Training Pads Lite,71353.37,20841.39,33.055852,0.030252


In [5]:
# 4. Check category-level story
sku_summary.groupby("category")[["revenue", "profit", "ad_spend"]].sum().sort_values("profit", ascending=False)

,revenue,profit,ad_spend
category,,,
Fitness,873621.08,123375.67,244550.81
Electronics,897837.63,122930.32,186429.02
Beauty,826126.14,101146.28,170322.70
Pet Supplies,710938.04,75427.62,164216.75
Home & Kitchen,558168.30,73177.96,127264.91


In [6]:
# 5. Monthly trend check
monthly = daily_performance.copy()
monthly["month"] = pd.to_datetime(monthly["date"]).dt.to_period("M").astype(str)

monthly.groupby("month")[["revenue", "profit", "ad_spend"]].sum()

,revenue,profit,ad_spend
month,,,
2025-01,670245.00,87425.63,154511.36
2025-02,597544.45,76592.00,138548.67
2025-03,664227.32,87475.46,153149.65
2025-04,628026.65,76903.63,147931.05
2025-05,662262.86,86821.74,150729.19
2025-06,644384.91,80839.39,147914.27
